# LLM Asset Classification Verification 

In [1]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer

data = pd.read_csv('test.csv')
data

,Entity_number,Description,Parent,Category,Class,Suspended,Tag,Valid_Class
0,NX5232A,"Chiller, Silo Bldg Control Room",TAB-ACC-CI,"H.V.A.C.,Chiller",HVAC,NaN,NaN,http://www.toronto.ca/TWONTO#chiller
1,TAB-WA1-SQ-1984,"Lanyard 6 Ft- Velasco, Gabriel\t\t\t\t\t\t\t",TAB-WA1-SQ-0001,"PPE,Harness",Safety Equipment,NaN,SQ,lifeline
2,THC-ACC-HTR-6025,"Heater, Unit, Electric, Heating System, Lower ...",THC-ELS-LP-4042A,"H.V.A.C.,Heater,Unit",HVAC,NaN,HTR,http://www.toronto.ca/TWONTO#heater
3,TAB-WA4-SQ-3430,Fall Limiter - MFLT2/705F,TAB-WA4-SQ-0010,"PPE,Lanyard",Safety Equipment,NaN,SQ,http://www.toronto.ca/TWONTO#fall_restricting_...
4,THC-ACC-PDIT-6291,"Transmitter, Pressure Differential, Filter F-6...",THC-ACC-F-6291,"Transmitter,Pressure",HVAC,NaN,PDIT,http://www.toronto.ca/TWONTO#pressure_transmitter
...,...,...,...,...,...,...,...,...
31752,YTF-SPC1,Process Control Systems - Administrative,YTF,NaN,NaN,NaN,NaN,http://www.toronto.ca/TWONTO#defined_collectio...
31753,YTF-SPC1,Process Control Systems - Administrative,YTF SPC,NaN,NaN,NaN,NaN,http://www.toronto.ca/TWONTO#defined_collectio...
31754,YTF-SS,Security Systems,YTF,NaN,NaN,NaN,NaN,http://www.toronto.ca/TWONTO#defined_collectio...
31755,YTF-STORE,Computer Storage Location,YTF EUS,NaN,NaN,NaN,NaN,http://www.toronto.ca/TWONTO#defined_collectio...


In [2]:
import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

In [3]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = data["Description"].dropna()  # Remove NaNs
sentences = sentences[sentences.apply(lambda x: isinstance(x, str))]  # Keep only strings
sentences = sentences.tolist()

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C

(31757, 384)
tensor([[1.0000, 0.2076, 0.3971,  ..., 0.2110, 0.2029, 0.2029],
        [0.2076, 1.0000, 0.0911,  ..., 0.1227, 0.0691, 0.0691],
        [0.3971, 0.0911, 1.0000,  ..., 0.2663, 0.1094, 0.1094],
        ...,
        [0.2110, 0.1227, 0.2663,  ..., 1.0000, 0.1871, 0.1871],
        [0.2029, 0.0691, 0.1094,  ..., 0.1871, 1.0000, 1.0000],
        [0.2029, 0.0691, 0.1094,  ..., 0.1871, 1.0000, 1.0000]])


In [6]:
first = []
second = []
for ridx, row in enumerate(similarities):
    for cidx, col in enumerate(row):
        if ridx != cidx and col > 0.95 and data['Valid_Class'][ridx] != data['Valid_Class'][cidx]:
            first.append([sentences[ridx],data['Valid_Class'][ridx], ridx])
            second.append([sentences[cidx], data['Valid_Class'][cidx], cidx])
        
    if ridx % 100 == 0:
        print(ridx)


0
100


KeyboardInterrupt: 

In [7]:
list(zip(first,second))

[(['Switch, Pressure-Low, Main Oil Pump, Blower 1911, Secondary Treatment, Aeration Blower System - North',
   'http://www.toronto.ca/TWONTO#pressure_switch',
   10],
  ['Switch, Level-Low, Auxiliary Oil Tank, Blower 1911, Secondary Treatment, Aeration Blower System - North',
   'http://www.toronto.ca/TWONTO#level_switch',
   4502]),
 (['Switch, Pressure-Low, Main Oil Pump, Blower 1911, Secondary Treatment, Aeration Blower System - North',
   'http://www.toronto.ca/TWONTO#pressure_switch',
   10],
  ['Switch, Level-Low, Auxiliary Oil Tank, Blower 1911, Secondary Treatment, Aeration Blower System - North',
   'http://www.toronto.ca/TWONTO#level_switch',
   11076]),
 (['Switch, Pressure-High, Pump 1315, Secondary Treatment, Aeration - Blower System - South',
   'http://www.toronto.ca/TWONTO#pressure_switch',
   27],
  ['Switch, Temperature-High, Pump 1315, Secondary Treatment, Aeration - Blower System - South',
   'http://www.toronto.ca/TWONTO#temperature_switch',
   2448]),
 (['Switch, 